### **Titanic - Machine Learning from Disaster**

#### **Environment initialization**

In [1]:
import numpy as np # Linear algebra
import pandas as pd # data processing, CSV file I/O (For example, pd.read_csv())

import os
for dirpath, dirnames, filenames in os.walk("../datasets"):
    print(f"Directory path of the dataset: {dirpath}")

    if dirnames != []:
        dirnames = ", ".join(dirnames)

    else:
        dirnames = "None"

    print(f"Subdirectories: {dirnames}")

    if filenames != []:
        filenames = ", ".join(filenames)

    else:
        filenames = "None"

    print(f"Datasets: {filenames}")

Directory path of the dataset: ../datasets
Subdirectories: None
Datasets: gender_submission.csv, test.csv, train.csv


`train.csv`

It contains the details of a subset of the passengers on board (891 passengers, to be exact -- where each passenger gets a different row in the table)

In [2]:
train_data = pd.read_csv("../datasets/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


`test.csv`

It contains the details to use for training the machine learning model to predict whether the other 418 passengers on board (in **test.csv**) survived.

In [3]:
test_data = pd.read_csv("../datasets/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


`gender_submission.csv`

An example that shows how you should structure your predictions. It predicts that all female passengers survived, and all male passengers died. Your hypotheses regarding survival will probably be different, which will lead to a different submission file. But, just like this file, your submission should have:

- a **"PassengerId"** column containing the IDs of each passenger from **test.csv**.
- a **"Survived"** column (that we will create!) with a "1" for the rows where you think the passenger survived, and a "0" where you predict that the passenger died.

In [4]:
gender_submissions_data = pd.read_csv("../datasets/gender_submission.csv")
gender_submissions_data.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


### **Explore a pattern**

Based on the `gender_submission.csv` we assume that all female passengers survived and all male passengers died.

Is this a reasonable pattern? We will check if this pattern holds true in the `train.csv`.

In [5]:
female_passengers = train_data.loc[train_data["Sex"] == "female"]["Survived"]

no_of_female_survivor = sum(female_passengers)
no_of_female_passenger = len(female_passengers)

pct_of_women_survivor = no_of_female_survivor / no_of_female_passenger

print(f"% of women who survived: {pct_of_women_survivor}")

% of women who survived: 0.7420382165605095


We compare our assumptions by checking also if all male passengers survived by comparing it to the percentage of female who survived

In [6]:
male_passengers = train_data.loc[train_data["Sex"] == "male"]["Survived"]

no_of_male_survivor = sum(male_passengers)
no_of_male_passenger = len(male_passengers)

pct_of_male_survivor = no_of_male_survivor / no_of_male_passenger

print(f"% of men who survived: {pct_of_male_survivor}")

% of men who survived: 0.18890814558058924


### **Random Forest Model**

We'll build what's known as a **random forest model**. This model is constructed of several "trees" (there are three trees in the picture below, but we'll construct 100!) that will individually consider each passenger's data and vote on whether the individual survived. Then, the random forest model makes a democratic decision: the outcome with the most votes wins!

![Random forest model](random_forest_model.png)

In [9]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('../outputs/submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
